# Февраль: доступные атрибуты для дашборда (логика техлида) + сравнение с Excel

Цель тетрадки:
- собрать все доступные на текущий момент атрибуты для дашборда за февраль 2026 по lake-логике техлида;
- явно зафиксировать, что не найдено / что не удалось посчитать;
- сравнить результат с Excel-отчетом за февраль и показать примеры строк `script vs excel`.


In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None


In [ ]:
# Параметры
month_start = '2026-02-01'
month_end = (pd.to_datetime(month_start) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

excel_path = '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx'

excel_cols_required = [
    'ИНН', 'Номер договора', 'ID договора', 'Наименование',
    'Количеств операций', 'Сумма операций', 'Ко-во торговых точек', 'Кол-во терминалов',
    'АУР', 'Амортизация', 'Комиссия (Р в месяц)', 'Комиссия \n(₽ в месяц)',
]

eps = 0.01

print('month_start =', month_start)
print('month_end =', month_end)


In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
with imp:
    imp.execute('invalidate metadata ods_alpha.scd1_agreements')
    imp.execute('invalidate metadata ods_alpha.scd1_companies')
    imp.execute('invalidate metadata ods_alpha.scd1_merchants')
    imp.execute('invalidate metadata ods_alpha.scd1_pos_terminals')
    imp.execute('invalidate metadata ods_alpha.scd1_trx')
    imp.execute('invalidate metadata ods_alpha.scd1_trx_acq')
    imp.execute('invalidate metadata sandbox_ai.kedr2hive__dbo__active_clients_ul_new')
    imp.execute('invalidate metadata ods.scd1_z_r2_ip_merchants')
    imp.execute('invalidate metadata ods.scd1_z_r2_tariff_tune')
    imp.execute('invalidate metadata ods.scd1_z_r2_tariff_fix')
print('Impala initialized')


In [ ]:
def normalize_numeric_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    s = re.sub(r"\D", '', s)
    return s if s else None

def normalize_inn(v):
    s = normalize_numeric_str(v)
    if s is None:
        return None
    if len(s) in (10, 12):
        return s
    if len(s) == 9:
        return s.zfill(10)
    if len(s) == 11:
        return s.zfill(12)
    return None

def normalize_agr_id(v):
    return normalize_numeric_str(v)

def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', ' ')
    s = re.sub(r"\s+", ' ', s)
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    return s.lower()

def normalize_num(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '')
    if s == '' or s.lower() in {'nan', 'none', 'null', '-'}:
        return None
    s = s.replace(',', '.')
    try:
        return float(Decimal(s))
    except Exception:
        return pd.to_numeric(s, errors='coerce')

def detect_excel_header_row(path, required_cols, scan_rows=40):
    raw = pd.read_excel(path, header=None)
    req = {str(c).strip() for c in required_cols}
    for i in range(min(scan_rows, len(raw))):
        vals = {str(x).strip() for x in raw.iloc[i].tolist()}
        if len(req & vals) >= 2:
            return i
    return 0


## 1) Excel-подготовка (февраль)


In [ ]:
header_row = detect_excel_header_row(excel_path, excel_cols_required)
excel_raw = pd.read_excel(excel_path, header=header_row)
excel_raw.columns = [str(c).strip() for c in excel_raw.columns]

fix_col = 'Комиссия (Р в месяц)' if 'Комиссия (Р в месяц)' in excel_raw.columns else ('Комиссия \n(₽ в месяц)' if 'Комиссия \n(₽ в месяц)' in excel_raw.columns else None)

excel_cols = {
    'ИНН': 'inn_raw',
    'Номер договора': 'contract_raw',
    'ID договора': 'agr_raw',
    'Наименование': 'name_excel',
    'Количеств операций': 'trx_cnt_excel',
    'Сумма операций': 'trx_sum_excel',
    'Ко-во торговых точек': 'retl_cnt_excel',
    'Кол-во терминалов': 'term_cnt_excel',
    'АУР': 'aur_excel',
    'Амортизация': 'amort_excel',
}
if fix_col is not None:
    excel_cols[fix_col] = 'fix_monthly_excel'

for c in excel_cols.keys():
    if c not in excel_raw.columns:
        print('WARNING: column not found in excel ->', c)

excel_use = excel_raw[[c for c in excel_cols.keys() if c in excel_raw.columns]].copy()
excel_use = excel_use.rename(columns=excel_cols)

excel_use['inn_norm'] = excel_use['inn_raw'].apply(normalize_inn)
excel_use['contract_norm'] = excel_use['contract_raw'].apply(normalize_contract)
excel_use['agr_id_norm'] = excel_use['agr_raw'].apply(normalize_agr_id)

for c in ['trx_cnt_excel', 'trx_sum_excel', 'retl_cnt_excel', 'term_cnt_excel', 'aur_excel', 'amort_excel', 'fix_monthly_excel']:
    if c in excel_use.columns:
        excel_use[c] = excel_use[c].apply(normalize_num)

excel_keys = excel_use.dropna(subset=['inn_norm', 'contract_norm']).copy()
excel_keys = excel_keys.drop_duplicates(subset=['inn_norm', 'contract_norm'])

display(pd.DataFrame([
    {'metric': 'excel_rows_total', 'value': len(excel_use)},
    {'metric': 'excel_rows_with_inn_contract', 'value': len(excel_keys)},
    {'metric': 'excel_unique_keys', 'value': excel_keys[['inn_norm', 'contract_norm']].drop_duplicates().shape[0]},
]))


## 2) Lake-слой атрибутов (доступные на текущий момент)


In [ ]:
sql_lake_base = f"""
with base as (
    select
        case
          when length(regexp_replace(trim(c.c_inn), '[^0-9]', '')) = 9 then lpad(regexp_replace(trim(c.c_inn), '[^0-9]', ''), 10, '0')
          when length(regexp_replace(trim(c.c_inn), '[^0-9]', '')) = 11 then lpad(regexp_replace(trim(c.c_inn), '[^0-9]', ''), 12, '0')
          else regexp_replace(trim(c.c_inn), '[^0-9]', '')
        end as inn_norm,
        cast(c.c_cmp_name as string) as client_name,
        lower(trim(cast(a.c_agr_number as string))) as contract_norm,
        cast(a.c_agr_number as string) as contract_number,
        cast(a.abs_agr_id as string) as agr_id_norm,
        cast(a.n_agr as string) as n_agr,
        cast(a.d_valid_from as date) as d_valid_from,
        cast(a.d_valid_to as date) as d_valid_to,
        a.n_cmp_client
    from ods_alpha.scd1_agreements a
    join ods_alpha.scd1_companies c
      on c.n_cmp = a.n_cmp_client
    where a.acq_class = 'SA'
      and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
      and coalesce(a.ods_deleted_flg, '0') <> '1'
      and coalesce(c.ods_deleted_flg, '0') <> '1'
),
retl as (
    select
        b.n_agr,
        count(distinct cast(m.c_nmrc as string)) as retl_cnt,
        min(cast(m.n_mcc as string)) as mcc_any,
        count(distinct cast(m.n_mcc as string)) as mcc_cnt
    from base b
    left join ods_alpha.scd1_merchants m
      on m.n_cmp = b.n_cmp_client
     and m.c_nmrc is not null
    group by b.n_agr
),
term as (
    select
        b.n_agr,
        count(distinct cast(t.c_nter as string)) as term_cnt
    from base b
    left join ods_alpha.scd1_merchants m
      on m.n_cmp = b.n_cmp_client
     and m.c_nmrc is not null
    left join ods_alpha.scd1_pos_terminals t
      on t.c_nmrc = m.c_nmrc
     and t.c_nter is not null
    group by b.n_agr
),
trx_f as (
    select
        t.n_trx,
        cast(t.n_amt_src as double) as n_amt_src
    from ods_alpha.scd1_trx t
    where cast(t.d_trans_date as date) >= cast('{month_start}' as date)
      and cast(t.d_trans_date as date) <= cast('{month_end}' as date)
      and t.c_nter is not null
),
trx_by_agr as (
    select
        cast(ta.n_agr as string) as n_agr,
        count(*) as trx_cnt,
        sum(tf.n_amt_src) as trx_sum
    from trx_f tf
    join ods_alpha.scd1_trx_acq ta
      on ta.n_trx = tf.n_trx
    group by cast(ta.n_agr as string)
),
fix_r2 as (
    select
        cast(m.id as string) as agr_id_norm,
        max(cast(tf.c_summa as double)) as fix_comiss,
        count(distinct cast(tf.c_summa as string)) as fix_variants_cnt
    from ods.scd1_z_r2_ip_merchants m
    left join ods.scd1_z_r2_tariff_tune tt
      on tt.c_tariff_plan = m.c_tariff_plan
    left join ods.scd1_z_r2_tariff_fix tf
      on tt.c_tariff = tf.id
    where m.id is not null
    group by cast(m.id as string)
),
ssp as (
    select
        case
          when length(regexp_replace(trim(cast(inn as string)), '[^0-9]', '')) = 9 then lpad(regexp_replace(trim(cast(inn as string)), '[^0-9]', ''), 10, '0')
          when length(regexp_replace(trim(cast(inn as string)), '[^0-9]', '')) = 11 then lpad(regexp_replace(trim(cast(inn as string)), '[^0-9]', ''), 12, '0')
          else regexp_replace(trim(cast(inn as string)), '[^0-9]', '')
        end as inn_norm,
        nullif(trim(cast(zo as string)), '') as zo
    from sandbox_ai.kedr2hive__dbo__active_clients_ul_new
    where inn is not null
),
ssp_card as (
    select
        inn_norm,
        min(zo) as zo_any,
        count(distinct zo) as zo_candidates
    from ssp
    group by inn_norm
)
select
    b.inn_norm,
    b.client_name,
    b.contract_norm,
    b.contract_number,
    b.agr_id_norm,
    b.n_agr,
    b.d_valid_from,
    b.d_valid_to,
    coalesce(r.retl_cnt, 0) as retl_cnt,
    coalesce(t.term_cnt, 0) as term_cnt,
    coalesce(x.trx_cnt, 0) as trx_cnt,
    coalesce(x.trx_sum, 0.0) as trx_sum,
    cast(coalesce(t.term_cnt, 0) as double) * 1926 as aur,
    r.mcc_any,
    coalesce(r.mcc_cnt, 0) as mcc_cnt,
    fr.fix_comiss,
    fr.fix_variants_cnt,
    sc.zo_any as ssp_zo,
    sc.zo_candidates
from base b
left join retl r on r.n_agr = b.n_agr
left join term t on t.n_agr = b.n_agr
left join trx_by_agr x on x.n_agr = b.n_agr
left join fix_r2 fr on fr.agr_id_norm = b.agr_id_norm
left join ssp_card sc on sc.inn_norm = b.inn_norm
"""

with imp:
    lake_df = imp.fetch(sql_lake_base)

lake_df['inn_norm'] = lake_df['inn_norm'].apply(normalize_inn)
lake_df['contract_norm'] = lake_df['contract_norm'].apply(normalize_contract)
lake_df['agr_id_norm'] = lake_df['agr_id_norm'].apply(normalize_agr_id)
for c in ['retl_cnt', 'term_cnt', 'trx_cnt', 'trx_sum', 'aur', 'mcc_cnt', 'fix_comiss', 'fix_variants_cnt', 'zo_candidates']:
    if c in lake_df.columns:
        lake_df[c] = pd.to_numeric(lake_df[c], errors='coerce')

display(pd.DataFrame([
    {'metric': 'lake_rows', 'value': len(lake_df)},
    {'metric': 'lake_unique_inn_contract', 'value': lake_df[['inn_norm', 'contract_norm']].drop_duplicates().shape[0]},
    {'metric': 'lake_rows_without_agr_id', 'value': int(lake_df['agr_id_norm'].isna().sum())},
]))
display(lake_df.head(20))


## 3) Что не найдено / что не удалось посчитать


In [ ]:
not_found_registry = pd.DataFrame([
    {
        'attribute': 'Филиал (РФ)',
        'status': 'not_in_current_script',
        'reason': 'Не добавлен стабильный join на branch-справочник в этой версии.'
    },
    {
        'attribute': 'ОГРН',
        'status': 'not_in_current_script',
        'reason': 'Источник `ods.scd1_z_cl_corp` не включен в текущую сборку.'
    },
    {
        'attribute': 'Тариф (наименование)',
        'status': 'partially_available',
        'reason': 'Есть `fix_comiss` из R2, но наименование тарифного плана в этой версии не подтянуто.'
    },
    {
        'attribute': 'Амортизация',
        'status': 'not_computed_here',
        'reason': 'Требует join на one-off таблицу амортизации терминалов, не включено в этот скрипт.'
    },
    {
        'attribute': 'Комиссия (% с операций)',
        'status': 'not_computed_here',
        'reason': 'В данной версии не подтягивалась tax-компонента из транзакционного слоя.'
    },
    {
        'attribute': 'Комиссия эквайринга (общая)',
        'status': 'not_computed_here',
        'reason': 'Нет полного набора компонент (процент + фикс) в единой методологии здесь.'
    },
    {
        'attribute': 'ЧОД / ФР / Средние IRF и т.д.',
        'status': 'not_computed_here',
        'reason': 'Требуются дополнительные бизнес-формулы и входы, не включенные в этот MVP-скрипт.'
    },
])

display(not_found_registry)


## 4) Сравнение с Excel: агрегаты и уровень строк


In [ ]:
# Сравнение по ключу INN + договор
cmp_df = excel_keys.merge(
    lake_df,
    on=['inn_norm', 'contract_norm'],
    how='left',
    suffixes=('_excel', '_lake')
)

cmp_df['has_lake_match'] = cmp_df['n_agr'].notna().astype(int)

for x, y, d in [
    ('trx_cnt_excel', 'trx_cnt', 'diff_trx_cnt'),
    ('trx_sum_excel', 'trx_sum', 'diff_trx_sum'),
    ('retl_cnt_excel', 'retl_cnt', 'diff_retl_cnt'),
    ('term_cnt_excel', 'term_cnt', 'diff_term_cnt'),
    ('aur_excel', 'aur', 'diff_aur'),
]:
    if x in cmp_df.columns and y in cmp_df.columns:
        cmp_df[d] = cmp_df[y] - cmp_df[x]

cmp_summary = pd.DataFrame([
    {'metric': 'excel_rows_for_compare', 'value': len(cmp_df)},
    {'metric': 'rows_matched_in_lake', 'value': int(cmp_df['has_lake_match'].sum())},
    {'metric': 'rows_not_matched_in_lake', 'value': int((cmp_df['has_lake_match'] == 0).sum())},
    {'metric': 'match_share', 'value': float(cmp_df['has_lake_match'].mean()) if len(cmp_df) else 0.0},
])

def eq_with_eps(s):
    return int((s.abs() <= eps).sum()) if len(s) else 0

metric_rows = []
for d in ['diff_trx_cnt', 'diff_trx_sum', 'diff_retl_cnt', 'diff_term_cnt', 'diff_aur']:
    if d in cmp_df.columns:
        s = pd.to_numeric(cmp_df[d], errors='coerce').dropna()
        metric_rows.append({
            'metric': d,
            'compared_rows': len(s),
            'matched_rows_eps': eq_with_eps(s),
            'matched_share_eps': (eq_with_eps(s) / len(s)) if len(s) else None,
            'mean_abs_diff': float(s.abs().mean()) if len(s) else None,
        })

metric_cmp_summary = pd.DataFrame(metric_rows)

display(cmp_summary)
display(metric_cmp_summary)


In [ ]:
print('Примеры строк script vs excel (matched):')
show_cols = [
    'inn_norm', 'contract_norm', 'name_excel', 'client_name', 'agr_id_norm_excel', 'agr_id_norm_lake',
    'trx_cnt_excel', 'trx_cnt', 'diff_trx_cnt',
    'trx_sum_excel', 'trx_sum', 'diff_trx_sum',
    'retl_cnt_excel', 'retl_cnt', 'diff_retl_cnt',
    'term_cnt_excel', 'term_cnt', 'diff_term_cnt',
    'aur_excel', 'aur', 'diff_aur',
    'ssp_zo', 'fix_comiss', 'fix_variants_cnt', 'mcc_any', 'mcc_cnt'
]
show_cols = [c for c in show_cols if c in cmp_df.columns]

display(cmp_df[cmp_df['has_lake_match'] == 1][show_cols].head(50))

print('Примеры строк без match в lake:')
display(cmp_df[cmp_df['has_lake_match'] == 0][[c for c in ['inn_norm', 'contract_norm', 'name_excel', 'agr_id_norm_excel'] if c in cmp_df.columns]].head(50))


## 5) Финальный комментарий

Тетрадка показывает:
- что уже доступно из lake-атрибутов по февральскому SA-периметру;
- что пока не найдено/не посчитано в текущем скрипте;
- насколько и где значения расходятся с Excel-референсом на уровне ключа `INN+договор`.
